# Event study: S&P 500 joiners and leavers

Reproduce CAR, volume and volatility plots using `src.events` and `src.utils.plotting`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.utils.config_loader import load_config
from src.data.load_data import load_events, build_ticker_permno_bridge
from src.data.preprocess_data import build_daily_panel
from src.events.event_study import run_event_study
import pandas as pd

In [ ]:
cfg = load_config()
panel_path = Path("data/interim/daily_panel.parquet")
if panel_path.exists():
    panel = pd.read_parquet(panel_path)
else:
    panel = build_daily_panel(config=cfg, max_chunks=20)
events = load_events(config=cfg)
bridge = build_ticker_permno_bridge(config=cfg)
events["permno"] = events["ticker"].map(bridge.groupby("ticker")["permno"].first().to_dict())
events = events.dropna(subset=["permno"]).copy()
events["permno"] = events["permno"].astype(int)

In [ ]:
result = run_event_study(panel, events, config=cfg)
result["car_by_rel"]